# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [9]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [10]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [11]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [12]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - 80c6a3b8


In [13]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [14]:
from langchain_core.tools import tool
from typing import List, Optional, Literal
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_symbol = {"pending": "[ ]", "in_progress": "[~]", "completed": "[x]"}
        symbol = status_symbol.get(todo["status"], "[?]")
        result.append(f"{symbol} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")


Todo tools defined!


In [15]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [16]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [17]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: d:\Projects\aie9\07_Deep_Agents\workspace


In [18]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
[FILE] comprehensive_morning_energy_guide.md (23321 bytes)
[FILE] morning_energy_routine_guide.md (28397 bytes)
[FILE] personalized_sleep_improvement_plan.md (5296 bytes)
[DIR] research
[DIR] workspace


In [19]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[FILE] sleep_notes.md (252 bytes)


In [20]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [21]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: d:\Projects\aie9\07_Deep_Agents\workspace


In [22]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Personalized Sleep Improvement Plan is Ready! 🌙

I've created a comprehensive, evidence-based sleep improvement plan tailored to your specific challenges and saved it to `/my_sleep_improvement_plan.md`. Here's what makes this plan special for your situation:

### Key Highlights:

**🎯 Addresses Your Specific Issues:**
- **Inconsistent bedtimes (10pm-1am)**: 3-phase gradual approach to stabilize your schedule
- **Phone use in bed**: Strategic elimination with replacement activities
- **Morning fatigue**: Focus on consistent wake times and morning light exposure

**📅 Realistic Timeline:**
- **8-12 weeks** total implementation
- **Gradual changes** every 2-3 days (research shows 73% success rate vs. 23% for overnight changes)
- **Phase-based approach** so you're not overwhelmed

**🔬 Evidence-Based Strategies:**
- Consistent wake time is more important than bedtime initially
- Morning light exposure within 1 hour of waking
- Phone removal from bedroom (can delay slee

In [23]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Analyze current sleep issues (completed)
✅ [todo_3] Research evidence-based sleep improvement strategies (completed)
✅ [todo_5] Create personalized sleep improvement plan (completed)
✅ [todo_7] Save plan to file for reference (completed)


Workspace contents:
  [FILE] comprehensive_morning_energy_guide.md (23321 bytes)
  [FILE] morning_energy_routine_guide.md (28397 bytes)
  [FILE] my_sleep_improvement_plan.md (6804 bytes)
  [FILE] personalized_sleep_improvement_plan.md (5296 bytes)
  [DIR] research/
  [FILE] sleep_improvement_research.md (10065 bytes)
  [DIR] workspace/


---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
Using to-do lists for planning involves a balance between structured execution and agility.

Explicit planning slows things down when the task is highly exploratory or short-lived; the time spent documenting the plan can exceed the time needed to simply execute the task. ALso, items should be granular enough to be actionable but broad enough to avoid "micro-management" loops. If they are too small, the agent gets stuck in administrative bloat; if too large, the agent may stall due to a lack of clear direction.
Additionally, if an agent generates todos without completing them, it creates context drift and "hallucinated progress," where the agent's internal state reflects a plan that doesn't match the actual state of the project, often leading to infinite loops or wasted compute.

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
A wellness agent should prioritize safety by placing critical user conditions, like allergies and medications, directly in the system prompt to ensure they are never forgotten or overlooked. The 16KB health document and historical metrics are better managed as external files accessed via retrieval, keeping the prompt clean while allowing the agent to reference specific data as needed. Core safety logic and emergency protocols must never be offloaded, as they serve as the essential guardrails for the agent's behavior.

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [24]:
# Activity #1: Build a Research Agent
from pathlib import Path

# Step 1 & 2: Create a tool to read the wellness guide from the data folder
@tool
def read_wellness_guide() -> str:
    """Read and return the contents of the Health Wellness Guide.
    
    Returns:
        The contents of data/HealthWellnessGuide.txt, or an error message if not found
    """
    guide_path = Path("data/HealthWellnessGuide.txt")
    if not guide_path.exists():
        return f"Health wellness guide not found at: {guide_path.resolve()}"
    return guide_path.read_text(encoding="utf-8")

# Step 3: Create the research agent with appropriate tools and system prompt
research_tools = [
    read_wellness_guide,
    write_todos,
    update_todo,
    list_todos,
    write_file,
    ls,
]

research_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=research_tools,
    backend=filesystem_backend,
    system_prompt="""You are a Wellness Research Agent specializing in evidence-based health strategies.

When given a research task:
1. Create a structured todo list to track your research progress
2. Use read_wellness_guide() to consult the HealthWellnessGuide.txt for relevant information
3. Research and compile at least 5 evidence-based strategies related to the topic
4. Save a comprehensive, well-formatted markdown report to the workspace
5. Update todos as you complete each research phase
6. Provide a clear summary highlighting key findings

Structure your reports with:
- Executive summary
- Evidence-based strategies (numbered list)
- Implementation guidelines
- Key takeaways
- Citations to the wellness guide where applicable

Be thorough, evidence-based, and practical."""
)

print("Research Agent created successfully!")

# Step 4: Test with the stress management research task
TODO_STORE.clear()

test_result = research_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies.

Please:
1. Consult the HealthWellnessGuide.txt for relevant information
2. Create a structured markdown report with clear sections
3. Save it to the workspace (e.g., workspace/stress_management_guide.md)
4. Provide a summary of what you found"""
    }]
})

print("\n" + "="*70)
print("AGENT RESPONSE")
print("="*70)
print(test_result["messages"][-1].content)

print("\n" + "="*70)
print("RESEARCH PROGRESS (TODOS)")
print("="*70)
print(list_todos.invoke({}))

print("\n" + "="*70)
print("WORKSPACE CONTENTS")
print("="*70)
print(ls.invoke({"path": "."}))

Research Agent created successfully!

AGENT RESPONSE
The comprehensive stress management guide is now complete and saved to your workspace. It provides 7 evidence-based strategies with clear implementation guidelines, making it a practical resource for both immediate stress relief and long-term resilience building.

RESEARCH PROGRESS (TODOS)
✅ [todo_1] Read the Health Wellness Guide (completed)
✅ [todo_3] Research evidence-based stress management strategies (completed)
✅ [todo_5] Create comprehensive markdown report (completed)
✅ [todo_7] Save report to workspace (completed)
✅ [todo_9] Provide executive summary (completed)

WORKSPACE CONTENTS
[FILE] comprehensive_morning_energy_guide.md (23321 bytes)
[FILE] morning_energy_routine_guide.md (28397 bytes)
[FILE] my_sleep_improvement_plan.md (6804 bytes)
[FILE] personalized_sleep_improvement_plan.md (5296 bytes)
[DIR] research
[FILE] sleep_improvement_research.md (10065 bytes)
[DIR] workspace

AGENT RESPONSE
The comprehensive stress manage

---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [25]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [26]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [27]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.
        
The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
Perfect! I've successfully created your comprehensive morning routine guide for better energy. Here's what I accomplished:

## ✅ Project Complete!

Your **"The Ultimate Morning Routine Guide: Transform Your Day with Science-Backed Energy Optimization"** is now ready and saved as `/comprehensive-morning-routine-guide.md`.

### What's Included:

🧠 **Science-Based Foundation**
- Circadian rhythm research and cortisol optimization
- Light exposure benefits and chronotype considerations
- Evidence-based habit formation principles

🏃‍♀️ **Exercise Section**
- HIIT, yoga, walking, and strength training options
- Detailed routines with timing recommendations
- Indoor/outdoor variations for all fitness levels

🥗 **Nutrition Guidance**
- Hydration strategies and balanced breakfast timing
- Energy-boosting foods vs. energy-draining choices
- Sample meal ideas for different time constraints

🧘‍♀️ **Mindset Practices**
- Meditation, journaling, and goal-setting techniques
- St

In [28]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines (completed)
✅ [todo_3] Research practical morning routine elements (completed)
✅ [todo_5] Create comprehensive morning routine guide (completed)
✅ [todo_7] Save guide as formatted markdown file (completed)

Generated files in workspace:
  [FILE] comprehensive-morning-routine-guide.md (14958 bytes)
  [FILE] comprehensive_morning_energy_guide.md (23321 bytes)
  [FILE] morning-routine-guide.md (33970 bytes)
  [FILE] morning_energy_routine_guide.md (28397 bytes)
  [FILE] my_sleep_improvement_plan.md (6804 bytes)
  [FILE] personalized_sleep_improvement_plan.md (5296 bytes)
  [DIR] research/
  [FILE] sleep_improvement_research.md (10065 bytes)
  [DIR] workspace/


## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [29]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [30]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [31]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [32]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Perfect! I can see from your profile that you're Alex, and your primary goal is to improve energy levels with better sleep as a secondary goal. I also note you prefer morning exercise and like detailed information. Given that you have mild anxiety, I'll recommend exercises that can help with both energy and anxiety management.

Here's a personalized exercise routine that aligns with your goals:

## **Morning Energy-Boosting Routine (4-5 days/week)**

### **Week 1-2: Foundation Building**
**Monday, Wednesday, Friday:**
- 5-10 minutes dynamic warm-up (arm circles, leg swings, gentle stretching)
- 15-20 minutes moderate cardio (brisk walking, light jogging, or cycling)
- 10 minutes bodyweight strength training:
  - Modified push-ups (knee or wall push-ups)
  - Bodyweight squats
  - Planks (start with 15-30 seconds)
- 5 minutes cool-down stretching

**Tuesday, Thursday:**
- 20-30 minutes gentle yoga or stretching routine (excellent for anxiety management)
- Focus on poses l

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [33]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [34]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [35]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [36]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [37]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Wellness Assessment Complete! 

Based on your profile as a 35-year-old office worker looking to lose 15 pounds while managing sleep issues and a sedentary lifestyle, here are my key findings:

### **Your Strengths:**
- Good baseline health with no major conditions
- Already making conscious dietary choices as a vegetarian  
- Clear, specific goals with realistic expectations
- At an optimal age for implementing lifestyle changes

### **Priority Areas to Address:**

**🔴 IMMEDIATE (Start Today):**
1. **Movement Integration** - Your sedentary work is the biggest barrier to weight loss
2. **Sleep Optimization** - Poor sleep disrupts hormones that control hunger and metabolism

**🟡 SHORT-TERM (Next 1-2 weeks):**
3. **Vegetarian Nutrition Optimization** - Leverage your diet for sustainable weight loss

### **Your Action Plan:**

**Start Today:**
- Set hourly reminders to stand/walk for 2-3 minutes
- Create a consistent bedtime routine (no phones in bedroom!)
- Begin t

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [38]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli not installed. Install with:
  uv pip install deepagents-cli
  # or
  pip install deepagents-cli


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [39]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [40]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [41]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
## 🎉 Your Complete Wellness Program is Ready!

I've created a comprehensive 2-week wellness program tailored specifically to your needs and preferences. Here's what you now have:

### 📁 Your Program Files:
- **`/alex_wellness_program.md`** - Master overview with daily integration tips
- **`/alex_exercise_plan.md`** - 3x/week, 30-minute workout routines with anxiety-friendly modifications
- **`/alex_nutrition_plan.md`** - Complete vegetarian meal plan with recipes and shopping lists
- **`/alex_mindfulness_plan.md`** - Daily stress management and sleep optimization strategies

### 🎯 Key Highlights:

**Exercise Program:**
- Functional fitness routines perfect for morning workouts
- Beginner-friendly with calming elements for anxiety management
- Progressive difficulty from Week 1 to Week 2

**Nutrition Plan:**
- 14 days of vegetarian meals designed to boost energy
- Includes anxiety-supporting foods and sleep-promoting nutrients
- Complete with recipes, shopping l

In [42]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Create personalized 2-week exercise routine (completed)
✅ [todo_3] Develop vegetarian meal plan (completed)
✅ [todo_5] Design stress management and sleep optimization program (completed)
✅ [todo_7] Create exercise plan file (completed)
✅ [todo_9] Create nutrition plan file (completed)
✅ [todo_11] Create wellness and mindfulness file (completed)
✅ [todo_13] Create master program overview (completed)

GENERATED FILES
  [FILE] alex_exercise_plan.md (5325 bytes)
  [FILE] alex_mindfulness_plan.md (7260 bytes)
  [FILE] alex_nutrition_plan.md (5770 bytes)
  [FILE] alex_wellness_program.md (5940 bytes)
  [FILE] comprehensive-morning-routine-guide.md (14958 bytes)
  [FILE] comprehensive_morning_energy_guide.md (23321 bytes)
  [FILE] morning-routine-guide.md (33970 bytes)
  [FILE] morning_energy_routine_guide.md (28397 bytes)
  [FILE] my_sleep_improvement_plan.md (6804 bytes)
  [FILE] personalized_sleep_improvement_plan.md (5296 bytes)
  [DIR] research/
  [FILE] slee

In [43]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of alex_exercise_plan.md:
# 2-Week Exercise Program for Alex

## Overview
- **Duration**: 2 weeks
- **Frequency**: 3 workouts per week
- **Session Length**: 30 minutes
- **Focus**: Functional fitness, energy improvement, stress relief
- **Emphasis**: Calming elements to reduce anxiety

---

## Week 1

### Day 1: Full Body Functional Workout
- **Purpose**: Build strength and energy levels
- **Format**: Circuit (perform each exercise for 30 seconds, rest 30 seconds, repeat circuit 2 times)

1. **Brisk Walk or March on the Spot**
   - **Description**: Walk or march in place at a faster pace.
   - **Tip**: Focus on breathing deeply to calm the mind.
   
2. **Bodyweight Squats**
   - **Modification**: Use a chair to assist for stability if necessary.
   - **Description**: Stand with feet shoulder-width apart, lower the hips back as if sitting on a chair, and return to standing.
   
3. **Wall Push-ups**
   - **Modification**: Perform on a table or counter for a lower angle.
   - **

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
Subagents should be granted distinct tools to prevent prompt confusion and ensure security, using shared tools only for basic utilities common to all tasks. Model selection follows the complexity of the role, where a high-reasoning model manages planning and coordination while smaller, faster models handle execution and formatting. The ideal granularity aligns with specific domain expertise; if an agent's instructions become overly complex, it should be split, but if subagents spend more time communicating than working, they should be consolidated.

## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
To move a wellness agent into production, we must implement strict safety guardrails using a dedicated moderation layer to block medical diagnoses and enforce disclaimers. The architecture must shift from in-memory processing to persistent storage to maintain long-term user history. For operational stability, we need comprehensive observability tools to track agent reasoning paths and cost management strategies, such as using token quotas and smaller models for routine subagent tasks, to prevent runaway expenses.

---
## 🏗️ Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [47]:
# Activity #2: Build a 30-Day Wellness Challenge System
# Uses all 4 Deep Agent elements: Planning, Context Management, Subagents, Memory

# All necessary imports 
from langchain_core.tools import tool
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from pathlib import Path
from typing import List, Literal
import os
import json
import sys
import io

# Set UTF-8 encoding for Windows compatibility
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except:
        pass

# Helper function to safely print content with emoji fallback
def safe_print(text):
    """Print text safely, replacing problematic Unicode characters if needed."""
    try:
        print(text)
    except UnicodeEncodeError:
        # Replace problematic unicode characters
        replacements = {
            '\U0001f31f': '*',  # Star emoji
            '\u2705': '[OK]',   # Checkmark
            '\u26a0': '[!]',    # Warning sign
            '\U0001f4aa': '[!]', # Flexed biceps
            '\U0001f3c6': '[*]', # Trophy
            '\u2728': '*',      # Sparkles
            '\U0001f1f3': '*',   # Various emojis
        }
        cleaned = text
        for emoji, replacement in replacements.items():
            cleaned = cleaned.replace(emoji, replacement)
        try:
            print(cleaned)
        except UnicodeEncodeError:
            # Last resort: encode with error handling
            print(text.encode('ascii', 'replace').decode('ascii'))

print("="*70)
print("ACTIVITY #2: 30-DAY WELLNESS CHALLENGE SYSTEM")
print("="*70)
print("\nNote: This activity uses tools from earlier cells.")
print("Make sure you've run all cells through Task 7 (Long-term Memory)\n")

# Step 1: Define specialized subagent configurations for the 30-day challenge

challenge_subagents = [
    {
        "name": "goal-tracker",
        "description": "Tracks and monitors daily progress against wellness goals. Provides motivation and feedback.",
        "system_prompt": """You are a 30-Day Wellness Challenge Goal Tracker. Your role is to:
1. Monitor daily progress against the user's specific goals
2. Calculate metrics (days completed, consistency %, goal progress)
3. Provide motivational feedback based on performance
4. Flag potential obstacles early
5. Celebrate milestones

Be encouraging but honest about progress. Use data to guide adjustments.""",
        "tools": [],
        "model": "openai:gpt-4o-mini",
    },
    {
        "name": "daily-adapter",
        "description": "Adapts recommendations based on feedback and adjusts the wellness plan dynamically.",
        "system_prompt": """You are a Daily Wellness Adapter. Your role is to:
1. Analyze user feedback and performance metrics
2. Identify what's working and what isn't
3. Suggest daily micro-adjustments to the plan
4. Provide alternative strategies if exercises/activities aren't working
5. Ensure the plan stays realistic and achievable

Be practical and flexible. Small tweaks often work better than major overhauls.""",
        "tools": [],
        "model": "openai:gpt-4o-mini",
    },
]

# Step 2: Create tools for managing daily check-ins and weekly summaries

@tool
def save_daily_checkin(day: int, user_id: str, notes: str, completed_tasks: int, total_tasks: int) -> str:
    """Save a daily check-in for the 30-day challenge.
    
    Args:
        day: Which day of the challenge (1-30)
        user_id: The user's unique identifier
        notes: User's notes about their day
        completed_tasks: Number of wellness tasks completed today
        total_tasks: Total wellness tasks for the day
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "challenge_checkins")
    checkin_data = {
        "day": day,
        "notes": notes,
        "completed": completed_tasks,
        "total": total_tasks,
        "completion_rate": (completed_tasks / total_tasks * 100) if total_tasks > 0 else 0
    }
    memory_store.put(namespace, f"day_{day}", checkin_data)
    return f"Saved check-in for Day {day}: {completed_tasks}/{total_tasks} tasks completed"

@tool
def get_challenge_progress(user_id: str) -> str:
    """Retrieve the user's 30-day challenge progress.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        Progress summary with statistics
    """
    namespace = (user_id, "challenge_checkins")
    checkins = list(memory_store.search(namespace))
    
    if not checkins:
        return "No progress data found. Start your challenge today!"
    
    total_days = len(checkins)
    total_completed = sum(checkin.value.get("completed", 0) for checkin in checkins)
    total_tasks = sum(checkin.value.get("total", 1) for checkin in checkins)
    completion_rate = (total_completed / total_tasks * 100) if total_tasks > 0 else 0
    
    result = [
        f"30-Day Challenge Progress for {user_id}:",
        f"  Days active: {total_days}/30",
        f"  Overall completion rate: {completion_rate:.1f}%",
        f"  Tasks completed: {total_completed}/{total_tasks}",
    ]
    return "\n".join(result)

safe_print("[OK] Daily check-in and progress tracking tools defined!")

# Step 3: Build the main 30-day challenge coordinator agent

# Verify that required tools from earlier cells are available
required_tools = [
    ("write_todos", write_todos),
    ("update_todo", update_todo),
    ("list_todos", list_todos),
    ("write_file", write_file),
    ("read_file", read_file),
    ("ls", ls),
    ("get_user_profile", get_user_profile),
    ("save_user_preference", save_user_preference),
]

missing_tools = []
available_tools = []
for tool_name, tool_obj in required_tools:
    try:
        if tool_obj is not None:
            available_tools.append(tool_obj)
        else:
            missing_tools.append(tool_name)
    except NameError:
        missing_tools.append(tool_name)

if missing_tools:
    safe_print(f"\n[WARNING] Missing tools from earlier cells: {', '.join(missing_tools)}")
    safe_print("Please run cells from Task 3-7 to define these tools.\n")

challenge_coordinator = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=available_tools + [save_daily_checkin, get_challenge_progress],
    backend=filesystem_backend,
    subagents=challenge_subagents,
    system_prompt="""You are a 30-Day Wellness Challenge Coordinator. Your mission is to guide users through a personalized 30-day wellness transformation.

## Your Responsibilities

1. **Initial Setup**: Get user profile and understand their goals, constraints, and preferences
2. **Challenge Generation**: Create a 30-day personalized plan with daily tasks across wellness pillars (exercise, nutrition, sleep, stress, mindfulness)
3. **Daily Management**: Track daily progress using todos and check-ins
4. **Progress Tracking**: Monitor completion rates and generate weekly summaries
5. **Adaptation**: Delegate to goal-tracker and daily-adapter subagents for analysis and adjustments
6. **Documentation**: Save plans, summaries, and progress reports to files

## Challenge Structure

Days 1-7: Foundation week - Establish baseline habits
Days 8-14: Building week - Increase intensity and complexity
Days 15-21: Momentum week - Deepen practices and add challenges
Days 22-30: Mastery week - Sustain and integrate all changes

## Daily Task Categories
- Physical Activity (30 mins minimum)
- Nutrition (3 healthy meals)
- Sleep Quality (consistent schedule)
- Stress Management (meditation/breathing)
- Mindfulness (reflection/gratitude)

## Key Rules
- Celebrate small wins
- Adapt the plan if needed (via daily-adapter)
- Track every day (save_daily_checkin)
- Generate weekly summaries (save to files)
- Remember user context from profile
- Never force unrealistic goals

## Output Files
- workspace/challenge_plan_[user_id].md - The complete 30-day plan
- workspace/weekly_summary_[user_id]_week[1-4].md - Weekly progress reports
- workspace/challenge_progress_[user_id].json - Complete metrics"""
)

safe_print("[OK] 30-Day Challenge Coordinator created!")

# Step 4: Test with a user creating their 30-day challenge

TODO_STORE.clear()

print("\n" + "="*70)
print("STARTING 30-DAY WELLNESS CHALLENGE")
print("="*70)

# First, initialize user profile for the challenge
user_challenge_id = "user_challenge_demo"
profile_namespace = (user_challenge_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Jordan"})
memory_store.put(profile_namespace, "goals", {
    "primary": "build consistent healthy habits",
    "secondary": ["lose 10 pounds", "improve sleep", "reduce stress"],
    "timeline": "30 days"
})
memory_store.put(profile_namespace, "constraints", {
    "time_available_daily": "45 minutes",
    "fitness_level": "intermediate",
    "dietary_preferences": ["no dairy", "vegetarian"],
    "medical_conditions": ["occasional lower back pain"]
})

safe_print(f"\n[OK] User profile created for {user_challenge_id}")

# Launch the challenge
safe_print("\nLaunching 30-day challenge...\n")

challenge_result = challenge_coordinator.invoke({
    "messages": [{
        "role": "user",
        "content": f"""I'm ready to start the 30-Day Wellness Challenge! My user_id is {user_challenge_id}.

Here's my situation:
- I have 45 minutes daily for wellness activities
- I want to build consistent healthy habits and lose 10 pounds
- I'm vegetarian with no dairy tolerance
- I have occasional lower back pain that I need to work around
- I want to improve my sleep and reduce stress

Please:
1. Create a personalized 30-day plan that respects my constraints
2. Save the complete plan to a file
3. Create initial todos for Week 1
4. Provide a summary of what I'll be doing"""
    }]
})

print("Challenge Coordinator Response:")
print("-" * 70)
safe_print(challenge_result["messages"][-1].content)

print("\n" + "="*70)
print("INITIAL WEEK 1 PLAN")
print("="*70)
safe_print(list_todos.invoke({}))

print("\n" + "="*70)
print("WORKSPACE FILES CREATED")
print("="*70)
try:
    safe_print(ls.invoke({"path": "."}))
except Exception as e:
    safe_print(f"(Could not list workspace: {e})")

# Step 5: Simulate a daily check-in and adaptation

print("\n" + "="*70)
print("SIMULATING DAY 3 CHECK-IN AND ADAPTATION")
print("="*70)

# Save a day 3 check-in
checkin_result = save_daily_checkin.invoke({
    "day": 3,
    "user_id": user_challenge_id,
    "notes": "Great day! Morning walk was energizing. Meditation felt a bit rushed. Back pain flared after yoga.",
    "completed_tasks": 4,
    "total_tasks": 5
})
safe_print(f"\n[OK] {checkin_result}")

# Get progress
progress_result = get_challenge_progress.invoke({
    "user_id": user_challenge_id
})
safe_print(f"\n{progress_result}")

# Request adaptation based on feedback
print("\n" + "="*70)
print("REQUESTING ADAPTATION FOR DAY 4")
print("="*70)

adaptation_result = challenge_coordinator.invoke({
    "messages": [{
        "role": "user",
        "content": f"""Day 3 check-in: I completed 4/5 tasks. The yoga triggered my lower back pain.
Meditation felt rushed. What should I adjust for Day 4 and beyond?

Please use the daily-adapter subagent to analyze what's working and suggest modifications.
Then save an adjusted plan for Days 4-7 considering this feedback."""
    }]
})

print("\nAdaptation Recommendation:")
print("-" * 70)
safe_print(adaptation_result["messages"][-1].content)

print("\n" + "="*70)
print("30-DAY CHALLENGE SIMULATION COMPLETE")
print("="*70)
print(f"\nUser {user_challenge_id} is now enrolled in the challenge with:")
print(f"  [OK] Personalized 30-day plan")
print(f"  [OK] Week 1 todos created")
print(f"  [OK] Progress tracking in place")
print(f"  [OK] Adaptation system ready for daily feedback")
print(f"  [OK] All 4 Deep Agent elements integrated:\n")
print(f"    1. Planning: Todo lists track daily tasks")
print(f"    2. Context Management: Plans & progress saved to files")
print(f"    3. Subagent Spawning: Goal-tracker & daily-adapter coordinate")
print(f"    4. Long-term Memory: User profile & check-ins stored")


ACTIVITY #2: 30-DAY WELLNESS CHALLENGE SYSTEM

Note: This activity uses tools from earlier cells.
Make sure you've run all cells through Task 7 (Long-term Memory)

[OK] Daily check-in and progress tracking tools defined!
[OK] 30-Day Challenge Coordinator created!

STARTING 30-DAY WELLNESS CHALLENGE

[OK] User profile created for user_challenge_demo

Launching 30-day challenge...

[OK] 30-Day Challenge Coordinator created!

STARTING 30-DAY WELLNESS CHALLENGE

[OK] User profile created for user_challenge_demo

Launching 30-day challenge...

Challenge Coordinator Response:
----------------------------------------------------------------------
## 🎉 Welcome to Your 30-Day Wellness Challenge, Jordan!

### What You'll Be Doing

Your challenge is perfectly tailored to your 45-minute daily commitment, vegetarian dairy-free lifestyle, and back pain considerations. Here's what to expect:

**🗓️ Week 1 (Foundation Week)**: You'll start gently with 20 minutes of back-safe stretches and walking, plus

---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)